# SocialChoice 08 - Kemeny, Ranked Pairs et Dodgson : trois regles computationnellement dures

**Navigation** : [Predecessor: [07-Committees-Core]] | [Index](../README.md)

**Autres notebooks** : [03-Voting-Methods.ipynb](03-Voting-Methods.ipynb) (regles polynomiales) | [04-Computational-Aggregation-SAT-Z3.ipynb](04-Computational-Aggregation-SAT-Z3.ipynb) (encodage SAT des theoremes d'impossibilite)

**Kernel** : Python 3

***

## Introduction

Le carnet SC-03 a presente les regles de vote **polynomiales** (pluralite, Borda, Copeland, IRV). Ce carnet-ci presente les **trois regles canoniques computationnellement dures** :

- **Kemeny** (Kemeny 1959 ; Handbook ch. 2 §2.7, ch. 4 §4.1-4.2) -- regle du **rang medianique** : choisir le classement qui maximise le nombre d'accords pairwise avec les preferences des electeurs (Kemeny score). Caracterisation axiomatique propre par Young-Levenglick (1978) : Kemeny est l'unique regle qui satisfait a la fois **Condorcet-coherence** et **neutralite**.
- **Ranked Pairs** (Tideman 1987 ; Handbook ch. 4 §4.5, Theoreme 4.15, Brill et Fischer 2012) -- trier les duels pairwise par marge decroissante, verrouiller les paires dans l'ordre tant qu'on ne cree pas de cycle (algorithme de Tideman). **Nuance de complexite** : construire le classement avec un **departage fixe** est polynomial (tri + verrouillage, ce que le carnet implemente) ; decider si une alternative est un gagnant sur l'**ensemble des departages** est **NP-complet** (Brill et Fischer 2012).
- **Dodgson** (Dodgson 1876 ; Handbook ch. 5, §5.3 Winner-Problem Complexity) -- pour chaque candidat x, le **Dodgson score** est le nombre minimal d'inversions de paires adjacentes dans les bulletins individuels necessaires pour faire de x le vainqueur de Condorcet. Le gagnant est celui de score minimum. Winner-Determination est **NP-dur** (Handbook ch. 5 §5.3 Winner-Problem Complexity, Caragiannis-E.Hemaspaandra-L.Hemaspaandra 2016).

Ces trois regles sont le **contrepoint computationnel** des regles polynomiales de SC-03 : un moteur SOTA (Z3, solveur ILP, ou tout SAT solver) prend tout son sens ici -- la complexite NP-dure est structurellement un probleme non trivial (axe 2 de #3801).

### Prerequis : SC-03 et SC-04

- SC-03 : cycle de Condorcet, duels pairwise, regles de vote classiques.
- SC-04 : encodage SAT/Z3 des theoremes d'impossibilite (Arrow, Sen).

### Duree estimee : 35 minutes


> *Ancres savantes -- Kemeny, J.G. (1959), Mathematics without numbers, Daedalus 88(4):577-591 (Kemeny score : classement qui maximise la somme des accords pairwise, la regle de Kemeny) ; Young, H.P. & Levenglick, A. (1978), A consistent extension of Condorcet's election principle, SIAM Journal on Applied Mathematics 35(2):285-300 (caracterisation axiomatique de Kemeny : unique regle a satisfaire Condorcet-coherence + neutralite, voir aussi Handbook ch. 4 §4.1-4.2) ; Tideman, T.N. (1987), Independence of clones as a criterion for voting rules, Social Choice and Welfare 4(3):185-206 (Ranked Pairs, Handbook ch. 4 §4.5, Theoreme 4.15, Brill et Fischer 2012) ; Dodgson, C. (1876), A method of taking votes upon more than two issues, manuscrit non publie, reimprime dans McLean, I. & Urken, A.B. (1995), Classics of Social Choice, University of Michigan Press, pp. 273-287 (Dodgson score, manuscrit redige par l'auteur d'Alice au pays des merveilles ; voir Handbook ch. 5 §5.3) ; Caragiannis, I., Hemaspaandra, E. & Hemaspaandra, L. (2016), Dodgson's Rule and Young's Rule, in Handbook of Computational Social Choice (Brandt et al. ed.), ch. 5, pp. 121-143, Cambridge University Press, ISBN 978-1-107-06043-2 (NP-durete de la determination du gagnant Dodgson, §5.3 Winner-Problem Complexity) ; Brandt, F., Conitzer, V., Endriss, U., Lang, J., Procaccia, A.D. (2016), Handbook of Computational Social Choice, Cambridge University Press, ISBN 978-1-107-06043-2 (ouvrage d'ancrage, ch. 4 §4.5 ranked pairs, ch. 2 §2.7 Kemeny regle, ch. 4 §4.1-4.2 Kemeny, ch. 5 §5.3 Dodgson et NP-durete, source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`).*



***

## 0. Configuration et imports

Ce carnet utilise `numpy` pour les manipulations de matrices et `itertools` pour l'enumeration des permutations (Kemeny est resolu par enumeration exhaustive des `m!` permutations possibles).

In [1]:
# Configuration et imports
import numpy as np
from itertools import permutations, combinations, product
from collections import defaultdict, Counter
import random

print("Configuration OK : SocialChoice 08 - Kemeny, Ranked Pairs, Dodgson")
print(f"numpy version : {np.__version__}")

Configuration OK : SocialChoice 08 - Kemeny, Ranked Pairs, Dodgson
numpy version : 2.5.4


### Configuration de l'environnement

Le carnet utilise principalement `numpy` pour les calculs de matrices pairwise et `itertools.permutations` pour enumerer les classements. Pour `m <= 6` alternatives, l'enumeration directe reste rapide ; au-dela, il faut basculer sur une formulation ILP/SAT, presentee en section 4.

***

## 1. Regle de Kemeny : Kemeny score exact par enumeration des permutations

### 1.1 Definition : Kemeny score

Pour un profil de preferences `P` sur un ensemble d'alternatives `A`, le **Kemeny score** d'un classement `sigma` (une permutation de `A`) est :

```
kemeny_score(sigma, P) = somme_{x avant y dans sigma} nombre de votants preferant x a y
```

Ce score compte les accords individuels, y compris lorsque les duels sont a egalite.

Le **Kemeny winner** est l'ensemble des alternatives `a` qui peuvent figurer en tete d'un classement de score maximum.

In [2]:
def pairwise_margin(profile, alternatives):
    """Calcule la matrice des marges pairwise N(a_i, a_j).

    N(a, b) = (nb votants preferant a > b) - (nb votants preferant b > a).
    Convention : N(a, b) > 0 signifie majorite pour a.
    """
    margins = {}
    for a, b in permutations(alternatives, 2):
        a_wins = sum(1 for pref in profile if pref.index(a) < pref.index(b))
        b_wins = sum(1 for pref in profile if pref.index(b) < pref.index(a))
        margins[(a, b)] = a_wins - b_wins
    return margins


def kemeny_score_of_ranking(sigma, profile, alternatives):
    """Nombre total d'accords entre le classement et les bulletins."""
    return sum(
        pref.index(x) < pref.index(y)
        for x, y in combinations(sigma, 2)
        for pref in profile
    )


def kemeny_bruteforce(profile, alternatives):
    """Kemeny winner exact par enumeration exhaustive des permutations.

    Pour |A| = m, il y a m! classements possibles. On prend celui de
    score Kemeny maximum, et on retient tous les candidats qui peuvent
    figurer en tete d'un tel classement optimal.

    Returns:
        (best_score, list de classements optimaux, set de winners)
    """
    m = len(alternatives)
    assert m <= 8, f"Enumeration prohibitive pour m={m} (>8 alternatives)"

    best_score = -float('inf')
    optimal_rankings = []
    for sigma in permutations(alternatives):
        s = kemeny_score_of_ranking(sigma, profile, alternatives)
        if s > best_score:
            best_score = s
            optimal_rankings = [sigma]
        elif s == best_score:
            optimal_rankings.append(sigma)

    winners = set(sigma[0] for sigma in optimal_rankings)
    return best_score, optimal_rankings, winners


# Test 1 : profil classique de Condorcet (cycle symetrique)
condorcet_profile = [
    ['A', 'B', 'C'],
    ['B', 'C', 'A'],
    ['C', 'A', 'B'],
]
alts = ['A', 'B', 'C']

score, optimal, winners = kemeny_bruteforce(condorcet_profile, alts)
print("PROFIL DE CONDORCET (cycle symetrique A > B > C > A)")
print("=" * 60)
for i, pref in enumerate(condorcet_profile):
    print(f"  Votant {i+1}: {' > '.join(pref)}")
print()
print(f"Kemeny score max : {score}")
print(f"Classements optimaux ({len(optimal)}) :")
for sigma in optimal:
    print(f"  {' > '.join(sigma)}")
print()
print(f"Kemeny winners : {sorted(winners)}")

PROFIL DE CONDORCET (cycle symetrique A > B > C > A)
  Votant 1: A > B > C
  Votant 2: B > C > A
  Votant 3: C > A > B

Kemeny score max : 5
Classements optimaux (3) :
  A > B > C
  B > C > A
  C > A > B

Kemeny winners : ['A', 'B', 'C']


### Interpretation : Kemeny sur le cycle de Condorcet

Les trois classements optimaux suivent chacun deux arcs du cycle. Leurs premiers candidats sont distincts : les trois candidats sont donc vainqueurs Kemeny. Le score compte les accords individuels sur toutes les paires.

L’enumeration examine `m!` classements. La formulation par optimisation evite de les enumerer explicitement, mais la determination d’un vainqueur Kemeny reste un probleme NP-dur.


In [3]:
# Test 2 : profil divergent de SC-03 (Borda et Copeland donnent B)
divergent_profile = [
    ['A', 'B', 'C'],
    ['A', 'B', 'C'],
    ['B', 'C', 'A'],
    ['C', 'B', 'A'],
    ['C', 'B', 'A'],
]

score_d, optimal_d, winners_d = kemeny_bruteforce(divergent_profile, alts)
print("PROFIL DIVERGENT (SC-03 : Borda + Copeland = B)")
print("=" * 60)
for i, pref in enumerate(divergent_profile):
    print(f"  Votant {i+1}: {' > '.join(pref)}")
print()
print(f"Kemeny score max : {score_d}")
print(f"Classements optimaux ({len(optimal_d)}) :")
for sigma in optimal_d:
    print(f"  {' > '.join(sigma)}")
print()
print(f"Kemeny winners : {sorted(winners_d)}")

PROFIL DIVERGENT (SC-03 : Borda + Copeland = B)
  Votant 1: A > B > C
  Votant 2: A > B > C
  Votant 3: B > C > A
  Votant 4: C > B > A
  Votant 5: C > B > A

Kemeny score max : 9
Classements optimaux (1) :
  B > C > A

Kemeny winners : ['B']


### Interpretation : Kemeny sur le profil divergent

Le classement optimal est B > C > A : B bat chacun de ses concurrents et C bat A. B est donc l'unique vainqueur Kemeny ; Borda et Copeland donnent aussi B sur ce profil.


***

## 2. Ranked Pairs (Tideman 1987)

### 2.1 Algorithme

1. Pour chaque paire `(x, y)`, calculer la marge `N(x, y)` (positive : x l'emporte sur y par majorite).
2. Trier les paires par marge **decroissante** (tie-break par ordre alphabetique pour reproductibilite).
3. Pour chaque paire dans l'ordre, **verrouiller** la paire dans le graphe oriente SAUF si cela cree un cycle.

Le graphe final est acyclique ; un tri topologique donne un classement compatible. Les vainqueurs sont ses sources (candidats sans arc entrant), qui peuvent etre multiples si des duels sont a egalite.

In [4]:
def creates_cycle(graph, start, target, alternatives):
    """Verifie si target peut atteindre start dans le graphe.

    Si oui, ajouter start -> target creerait un cycle.
    """
    visited = set()
    stack = [target]
    while stack:
        node = stack.pop()
        if node == start:
            return True
        if node in visited:
            continue
        visited.add(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                stack.append(neighbor)
    return False


def ranked_pairs(profile, alternatives):
    """Regle des Ranked Pairs (Tideman 1987).

    Returns:
        (ordering, winners) :
        - ordering : extension topologique du graphe verrouille (liste)
        - winners : ensemble des sources (candidats sans arc entrant)
    """
    margins = pairwise_margin(profile, alternatives)

    # Paires avec marge strictement positive, triees par marge decroissante
    # tie-break : x puis y par ordre alphabetique pour determinisme
    pairs_sorted = []
    for x, y in permutations(alternatives, 2):
        mxy = margins[(x, y)]
        if mxy > 0:
            pairs_sorted.append((mxy, x, y))
    pairs_sorted.sort(key=lambda t: (-t[0], t[1], t[2]))

    # Construction du graphe : verrouiller chaque paire si pas de cycle
    graph = defaultdict(set)
    in_degree = {a: 0 for a in alternatives}

    for mxy, x, y in pairs_sorted:
        if creates_cycle(graph, x, y, alternatives):
            continue
        graph[x].add(y)
        in_degree[y] += 1

    # Tri topologique deterministe
    visited = set()
    order = []

    def dfs(node):
        if node in visited:
            return
        visited.add(node)
        for neighbor in sorted(graph[node]):
            dfs(neighbor)
        order.append(node)

    for a in sorted(alternatives):
        dfs(a)

    order.reverse()

    winners = {a for a in alternatives if in_degree[a] == 0}
    return order, winners


# Test sur le profil divergent
order_rp, winners_rp = ranked_pairs(divergent_profile, alts)
print("PROFIL DIVERGENT : Ranked Pairs")
print("=" * 60)
for i, pref in enumerate(divergent_profile):
    print(f"  Votant {i+1}: {' > '.join(pref)}")
print()
print(f"Ranked Pairs order : {' > '.join(order_rp)}")
print(f"Ranked Pairs winners : {sorted(winners_rp)}")

# Test sur le profil cyclique de Condorcet
order_rp_c, winners_rp_c = ranked_pairs(condorcet_profile, alts)
print()
print("PROFIL DE CONDORCET : Ranked Pairs")
print("=" * 60)
for i, pref in enumerate(condorcet_profile):
    print(f"  Votant {i+1}: {' > '.join(pref)}")
print()
print(f"Ranked Pairs order : {' > '.join(order_rp_c)}")
print(f"Ranked Pairs winners : {sorted(winners_rp_c)}")

PROFIL DIVERGENT : Ranked Pairs
  Votant 1: A > B > C
  Votant 2: A > B > C
  Votant 3: B > C > A
  Votant 4: C > B > A
  Votant 5: C > B > A

Ranked Pairs order : B > C > A
Ranked Pairs winners : ['B']

PROFIL DE CONDORCET : Ranked Pairs
  Votant 1: A > B > C
  Votant 2: B > C > A
  Votant 3: C > A > B

Ranked Pairs order : A > B > C
Ranked Pairs winners : ['A']


### Interpretation : Ranked Pairs sur les deux profils

Sur le profil divergent, les trois majorites sont verrouillees et B est la seule source. Sur le cycle, le departage alphabetique verrouille A > B puis B > C ; C > A est refuse car cet arc fermerait un cycle. A est alors la seule source. Un autre departage des marges egales peut choisir une autre source.

Ranked Pairs respecte le vainqueur de Condorcet : sur le profil divergent, B bat A et C et reste l'unique source.


***

## 3. Regle de Dodgson (1876)

### 3.1 Definition : Dodgson score

Pour un profil `P` et un candidat `x`, le **Dodgson score** `DS(x, P)` est le nombre minimal d'inversions de paires adjacentes dans les preferences individuelles (cumule sur tous les electeurs) necessaire pour faire de `x` le vainqueur de Condorcet sur le profil modifie.

Le **Dodgson winner** est l'ensemble des candidats `x` de score minimum.

### 3.2 Implementation directe

Pour chaque candidat et chaque bulletin, il suffit de considerer sa remontee par swaps adjacents : les autres swaps ne peuvent ameliorer ses duels. Une programmation dynamique conserve le cout minimal pour chaque vecteur de victoires, chaque coordonnee etant plafonnee au seuil de majorite stricte. Son espace croît exponentiellement avec le nombre de concurrents : cette methode convient aux petits profils.

In [5]:
def check_condorcet_winner_for(candidate, profile, alternatives):
    """Verifie si candidate est Condorcet winner dans profile."""
    for other in alternatives:
        if other == candidate:
            continue
        c_wins = sum(1 for p in profile if p.index(candidate) < p.index(other))
        if c_wins <= len(profile) / 2:
            return False
    return True


def dodgson_score(candidate, profile, alternatives):
    """Nombre minimal de swaps adjacents pour rendre candidate Condorcet.

    Pour chaque bulletin, seule la remontee du candidat peut ameliorer ses
    duels. La programmation dynamique conserve le moindre cout pour chaque
    vecteur de victoires, borne au seuil de majorite stricte.
    """
    opponents = [a for a in alternatives if a != candidate]
    threshold = len(profile) // 2 + 1
    states = {(0,) * len(opponents): 0}
    for pref in profile:
        position = pref.index(candidate)
        next_states = {}
        for shift in range(position + 1):
            new_position = position - shift
            wins = tuple(int(pref.index(a) >= new_position) for a in opponents)
            for counts, cost in states.items():
                updated = tuple(min(threshold, c + w) for c, w in zip(counts, wins))
                new_cost = cost + shift
                next_states[updated] = min(next_states.get(updated, float('inf')), new_cost)
        states = next_states
    return states.get((threshold,) * len(opponents), -1)


def dodgson_bruteforce(profile, alternatives):
    """Dodgson winner exact par programmation dynamique."""
    scores = {}
    for candidate in alternatives:
        s = dodgson_score(candidate, profile, alternatives)
        scores[candidate] = s

    valid_scores = [s for s in scores.values() if s >= 0]
    if not valid_scores:
        return -1, dict(scores), set()
    min_score = min(valid_scores)
    winners = {a for a, s in scores.items() if s == min_score}
    return min_score, scores, winners


# Test 1 : profil divergent
print("DODGSON SUR LE PROFIL DIVERGENT")
print("=" * 60)
for i, pref in enumerate(divergent_profile):
    print(f"  Votant {i+1}: {' > '.join(pref)}")
print()
ds_score, ds_all, ds_winners = dodgson_bruteforce(divergent_profile, alts)
for c, s in ds_all.items():
    print(f"  Dodgson score({c}) = {s}")
print()
print(f"Dodgson winners : {sorted(ds_winners)}")

# Test 2 : profil de Condorcet
print()
print("DODGSON SUR LE PROFIL DE CONDORCET")
print("=" * 60)
ds_c, ds_all_c, ds_winners_c = dodgson_bruteforce(condorcet_profile, alts)
for c, s in ds_all_c.items():
    print(f"  Dodgson score({c}) = {s}")
print()
print(f"Dodgson winners : {sorted(ds_winners_c)}")

DODGSON SUR LE PROFIL DIVERGENT
  Votant 1: A > B > C
  Votant 2: A > B > C
  Votant 3: B > C > A
  Votant 4: C > B > A
  Votant 5: C > B > A

  Dodgson score(A) = 2
  Dodgson score(B) = 0
  Dodgson score(C) = 1

Dodgson winners : ['B']

DODGSON SUR LE PROFIL DE CONDORCET
  Dodgson score(A) = 1
  Dodgson score(B) = 1
  Dodgson score(C) = 1

Dodgson winners : ['A', 'B', 'C']


### Interpretation : Dodgson winner sur les deux profils

**Profil divergent** : B est le Dodgson winner (score 0, deja Condorcet winner). C a besoin d'un swap et A de deux swaps pour devenir Condorcet winner. C'est coherent avec toutes les autres regles (Borda, Copeland, Kemeny, RP) : **B** est l'unique gagnant.

**Profil de Condorcet** : aucun candidat n'est Condorcet winner au depart. Le Dodgson score est le nombre minimal de swaps pour chacun :
- A : dans B > C > A, echanger C et A donne B > A > C.
- B : dans A > B > C, echanger A et B donne B > A > C.
- C : dans A > B > C, echanger B et C donne A > C > B.

Dans chaque cas, le candidat modifie gagne alors ses deux duels.

Les trois candidats ont le meme Dodgson score 1, donc tous sont Dodgson winners (egalite etalee, le cycle est symetrique). C'est coherent avec l'indecision attendue sur un cycle pur.

**Note sur la complexite** : les etats memorisent les victoires contre chaque concurrent, plafonnees au seuil de majorite. Leur nombre peut croitre exponentiellement avec le nombre d'alternatives ; la methode est destinee aux petites instances.


***

## 4. Confrontation Z3 du Kemeny winner (esprit de SC-04)

Pour mettre en valeur un solveur, il faut un probleme **non trivial**. Nous encodons le Kemeny winner comme un probleme d'optimisation Z3 : trouver la permutation sigma qui maximise le Kemeny score. Enumeration classique = `m!` permutations ; formulation Z3 = une optimisation globale puis une optimisation par candidat.

In [6]:
# Encodage Z3 du Kemeny winner
from z3 import Solver, Int, If, sat, Sum, Optimize

print("z3 disponible pour encodage Kemeny winner")

z3 disponible pour encodage Kemeny winner


### Implementation Z3 : Kemeny winner comme optimisation

On utilise `z3.Optimize` (solveur d'optimisation SMT) pour trouver le Kemeny score maximum sur des alternatives permutees. Le modele z3 donne la permutation gagnante. Pour chaque candidat, une optimisation supplementaire verifie s'il peut mener un classement optimal.

In [7]:
def kemeny_z3(profile, alternatives):
    """Kemeny winner exact via Z3 Optimize.

    Encode chaque permutation sigma comme des variables entieres de rang,
    et maximise le Kemeny score.

    Returns:
        (status, optimal_score, optimal_ranking, kemeny_winners)
    """
    m = len(alternatives)
    if m > 6:
        return ('TIMEOUT', None, None, None)

    # La somme compte les accords avec chaque bulletin, pas seulement
    # le sens de la majorite : les marges peuvent avoir des poids differents.
    rank = {a: Int(f'r_{a}') for a in alternatives}
    constraints = [rank[a] >= 0 for a in alternatives]
    constraints += [rank[a] < m for a in alternatives]
    constraints += [rank[a] != rank[b] for a, b in combinations(alternatives, 2)]
    score = Sum([
        If(rank[x] < rank[y],
           sum(pref.index(x) < pref.index(y) for pref in profile),
           sum(pref.index(y) < pref.index(x) for pref in profile))
        for x, y in combinations(alternatives, 2)
    ])

    def solve(first=None):
        opt = Optimize()
        opt.add(*constraints)
        if first is not None:
            opt.add(rank[first] == 0)
        opt.maximize(score)
        if opt.check() != sat:
            return None
        model = opt.model()
        return (model.eval(score).as_long(),
                sorted(alternatives, key=lambda a: model.eval(rank[a]).as_long()))

    result = solve()
    if result is None:
        return ('UNKNOWN', None, None, None)
    optimal_score, ranking = result
    winners = set()
    for alt in alternatives:
        constrained = solve(alt)
        if constrained is None:
            return ('UNKNOWN', None, None, None)
        if constrained[0] == optimal_score:
            winners.add(alt)
    return ('OPTIMAL', optimal_score, ranking, winners)


# Test 1 : profil divergent
print("KEMENY VIA Z3 (profil divergent)")
print("=" * 60)
result_z3 = kemeny_z3(divergent_profile, alts)
print(f"Status : {result_z3[0]}")
if result_z3[0] == 'OPTIMAL':
    print(f"Score optimal : {result_z3[1]}")
    print(f"Classement optimal : {' > '.join(result_z3[2])}")
    print(f"Kemeny winners : {sorted(result_z3[3])}")

# Test 2 : profil de Condorcet
print()
print("KEMENY VIA Z3 (profil de Condorcet)")
print("=" * 60)
result_z3_c = kemeny_z3(condorcet_profile, alts)
print(f"Status : {result_z3_c[0]}")
if result_z3_c[0] == 'OPTIMAL':
    print(f"Score optimal : {result_z3_c[1]}")
    print(f"Classement optimal : {' > '.join(result_z3_c[2])}")
    print(f"Kemeny winners : {sorted(result_z3_c[3])}")

KEMENY VIA Z3 (profil divergent)


Status : OPTIMAL
Score optimal : 9
Classement optimal : B > C > A
Kemeny winners : ['B']

KEMENY VIA Z3 (profil de Condorcet)
Status : OPTIMAL
Score optimal : 5
Classement optimal : B > C > A
Kemeny winners : ['A', 'B', 'C']


### Interpretation : Kemeny via Z3 sur les deux profils

L'enumeration et Z3 trouvent le meme score optimal et le meme ensemble de vainqueurs pour chacun des deux profils. Sur le profil divergent, B est seul vainqueur. Sur le cycle, A, B et C peuvent chacun mener un classement optimal. Le classement affiche par Z3 est un representant optimal ; il peut varier selon le solveur.

Pour verifier chaque candidat, on maximise a nouveau le score en imposant sa premiere place. Il est vainqueur exactement lorsque cet optimum contraint egale l'optimum global.


***

## 5. Resume

Ce carnet a presente les trois regles computationnellement dures du choix social (Kemeny NP-dur, Ranked Pairs polynomial avec departage fixe mais NP-complet sur l'ensemble des departages, Dodgson NP-dur) et les a confrontees a une formulation Z3. Les implementations Python sont executees (Kemeny par enumeration des permutations, Ranked Pairs par algorithme de Tideman, Dodgson par programmation dynamique sur les victoires pairwise). La formulation Z3 du Kemeny winner utilise `z3.Optimize` et confirme le resultat brut-force sur les deux profils tests.

| Regle | Complexite winner determination | Source |
|---|---|---|
| Kemeny | NP-dur (exact par enumeration : `m!` permutations) | Kemeny 1959, Handbook ch. 4 §4.1-4.2 |
| Ranked Pairs | **Polynomial** (tri + verrouillage, departage fixe) ; **NP-complet** pour decider si une alternative est gagnante sur l'ensemble des departages | Tideman 1987, Handbook ch. 4 §4.5 (Thm 4.15, Brill-Fischer 2012) |
| Dodgson | NP-dur (programmation dynamique sur les victoires pour les petits profils) | Dodgson 1876, Handbook ch. 5 §5.3, Caragiannis-E.Hemaspaandra-L.Hemaspaandra 2016 |

**Ancrage bibliographique** : le Handbook of Computational Social Choice (Brandt-Conitzer-Endriss-Lang-Procaccia, 2016, ISBN 978-1-107-06043-2) est la reference canonique pour ces trois regles. Source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`. Les trois regles presentent un **probleme non trivial** (axe 2 de #3801) : la determination du gagnant est structurellement NP-dure (Kemeny, Dodgson), ce qui fait du solveur un outil preuve du contenu - et non un gadget.

***

## Exercices

**Note pedagogique** : les exercices suivent la convention `pass` / `return None` / `print("Exercice a completer")` (regle C.1), pas de `raise NotImplementedError`.

### Exemple guide : generation d'un profil aleatoire

**Contexte** : `random_profile` tire, pour `n` electeurs, un classement aleatoire de toutes les `m` alternatives (sans remplacement, via `random.sample`). Le profil ainsi produit est la matiere premiere de l'exercice 1 ci-dessous : chaque electeur est une liste de classements, de la preference la plus forte a la plus faible.

In [8]:
def random_profile(n_voters, alternatives, seed=None):
    """Genere un profil aleatoire de preferences.

    Args:
        n_voters: nombre d electeurs
        alternatives: liste des alternatives
        seed: graine aleatoire (optionnel)

    Returns:
        list de listes : preferences aleatoires
    """
    if seed is not None:
        random.seed(seed)
    return [random.sample(alternatives, len(alternatives)) for _ in range(n_voters)]


### Exercice 1 : score de Kemeny sur des profils aleatoires

**Objectif** : a partir de `random_profile` (exemple guide ci-dessus) et de `kemeny_bruteforce`, ecrire `kemeny_scores_random(n_voters, alternatives, n_profiles, seed)` qui tire `n_profiles` profils aleatoires, calcule pour chacun le score de Kemeny optimal (le premier element du triplet rendu par `kemeny_bruteforce`) et retourne la liste de ces scores.

**Critere de verification** : chaque score est un entier entre 0 et `n_voters * m(m-1)/2`, et deux appels avec le meme `seed` rendent la meme liste.

In [9]:
# Exercice 1 : score de Kemeny sur des profils aleatoires
def kemeny_scores_random(n_voters, alternatives, n_profiles, seed=None):
    """Score de Kemeny optimal sur n_profiles profils aleatoires.

    Returns:
        list[int] : un score de Kemeny par profil tire
    """
    return None  # TODO etudiant


print("Exercice 1 a completer")

Exercice 1 a completer


#### Indication pour l'exercice 1

La generation aleatoire utilise `random.sample(alternatives, k)` qui produit une permutation aleatoire sans remplacement. Pour `m=4`, l'enumeration Kemeny necessite `4! = 24` permutations, instantane. Avec `seed=42` (par exemple), le profil doit etre reproductible -- utiliser le meme seed pour reproduire les resultats.

### Exercice 2 : Ranked Pairs vs Copeland

**Objectif** : Implementer la regle Copeland (deja presente dans SC-03) et comparer avec Ranked Pairs sur une serie de profils tests. Identifier les cas ou les deux regles donnent le meme vainqueur, et les cas ou elles divergent.

In [10]:
def copeland_rule(profile):
    """Regle de Copeland (reimplementation succincte).

    Pour chaque paire (x, y), winner est celui qui l'emporte par majorite.
    Le score Copeland d'un candidat = (nb victoires) - (nb defaites).
    """
    print("Exercice 2 a completer")
    return None


def compare_rp_vs_copeland(profile, alternatives):
    """Compare Ranked Pairs et Copeland sur un profil donne."""
    rp_order, rp_winners = ranked_pairs(profile, alternatives)
    cp_order = copeland_rule(profile)
    cp_winner = cp_order[0] if cp_order else None
    return {
        'rp_winner': sorted(rp_winners),
        'cp_winner': cp_winner,
        'agree': cp_winner in rp_winners,
    }


# Tests : remplacer par des profils ou les regles divergent
# Indice : Copeland ne respecte pas forcement IIA, donc sur des profils
# construits specialement, les deux regles peuvent diverger.
result_1 = copeland_rule(divergent_profile)
print(f"Copeland order (test) : {result_1}")

Exercice 2 a completer
Copeland order (test) : None


### Exercice 3 : Dodgson winner sur un profil aleatoire

**Objectif** : Comparer Dodgson et Kemeny sur des profils aleatoires de 4 alternatives et identifier les cas ou les vainqueurs coincident.

In [11]:
def exercise_3_dodgson_vs_kemeny():
    """Exercice 3 : Dodgson vs Kemeny sur profils aleatoires 4x4.

    1. Generer 5 profils aleatoires (4 electeurs, 4 alternatives, seed=42)
    2. Calculer Kemeny winner (brute-force enumeration, 4! = 24 permutations)
    3. Calculer Dodgson winner (programmation dynamique)
    4. Comparer les resultats
    """
    pass


# Tests : a executer apres implementation
exercise_3_dodgson_vs_kemeny()

#### Indication pour l'exercice 3

Pour `m=4` alternatives, l'enumeration Kemeny necessite `4! = 24` permutations, instantane. Pour Dodgson, la programmation dynamique conserve des vecteurs de victoires dont le nombre augmente avec les electeurs et les alternatives. Les Kemeny winners et Dodgson winners coincident sur certaines classes de profils (ceux ou Kemeny winner est deja Condorcet winner, score Dodgson = 0) mais peuvent diverger sur d'autres (voir Handbook ch. 5 §5.4 pour les exemples theoriques).

***

## References bibliographiques

- Kemeny, J.G. (1959). Mathematics without numbers. Daedalus 88(4):577-591.
- Young, H.P. & Levenglick, A. (1978). A consistent extension of Condorcet's election principle. SIAM Journal on Applied Mathematics 35(2):285-300.
- Tideman, T.N. (1987). Independence of clones as a criterion for voting rules. Social Choice and Welfare 4(3):185-206.
- Dodgson, C. (1876). A method of taking votes upon more than two issues. Manuscrit non publie, reimprime dans McLean, I. & Urken, A.B. (1995), Classics of Social Choice, University of Michigan Press, pp. 273-287.
- Caragiannis, I., Hemaspaandra, E. & Hemaspaandra, L. (2016). Dodgson's Rule and Young's Rule. In *Handbook of Computational Social Choice* (Brandt, Conitzer, Endriss, Lang & Procaccia ed.), ch. 5, pp. 121-143. Cambridge University Press. **Ancrage du carnet pour Dodgson** : §5.3 Winner-Problem Complexity. (L'article *Computational Social Choice Meets Databases* existe, mais il est de Kimelfeld, Kolaitis et Stoyanovich, IJCAI 2018, p. 317-323, et porte sur le vote dans un contexte de bases de donnees -- il n'est pas l'ancre Dodgson.)
- Brill, M. & Fischer, F. (2012). The Price of Neutrality for the Ranked Pairs Method. AAAI. **Ancrage du carnet pour la nuance NP-completude Ranked Pairs** : Theoreme 4.15, Handbook ch. 4 §4.5.
- Brandt, F., Conitzer, V., Endriss, U., Lang, J., Procaccia, A.D. (2016). *Handbook of Computational Social Choice*. Cambridge University Press. ISBN 978-1-107-06043-2. Source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`.


> **Lien avec SC-03 et SC-04** : ce carnet complete la ferme des regles de vote presentee dans SC-03 par les regles Kemeny, Ranked Pairs et Dodgson. SC-04 presente l'encodage SAT/Z3 des theoremes d'impossibilite (Arrow, Sen) ; la section 4 de ce carnet transpose cette approche a l'optimisation Kemeny. Les trois carnets (SC-03 polynomial, SC-04 SAT/Z3 theoremes, SC-08 regles computationnelles) forment le triplet pedagogique du cluster.

***

## À retenir

- Kemeny maximise les accords individuels ; ses vainqueurs sont les candidats en tête d’au moins un classement optimal.
- Ranked Pairs verrouille les majorités par marge décroissante sans créer de cycle ; le départage des marges égales peut compter.
- Dodgson minimise les swaps adjacents nécessaires pour obtenir un vainqueur de Condorcet. La programmation dynamique présentée convient aux petits profils.

## Vérifiez votre compréhension

1. Pourquoi faut-il optimiser de nouveau pour chaque candidat dans la méthode Z3 ?
2. Pourquoi Ranked Pairs refuse-t-il un arc qui ferme un cycle ?
3. Pourquoi suffit-il de faire monter le candidat pour calculer son score Dodgson ?

<details>
<summary>Éléments de réponse</summary>

1. Un autre classement optimal peut placer ce candidat en tête, même s’il n’est pas en tête du premier modèle trouvé.
2. Le graphe verrouillé doit rester acyclique pour admettre un classement compatible.
3. Les swaps entre autres candidats ne changent aucun de ses duels et ne peuvent donc améliorer son score.

</details>

## Pour aller plus loin

Comparez les règles sur les profils de SC-03 et examinez les différentes possibilités de départage de Ranked Pairs. Les références ci-dessus donnent les résultats de complexité associés.


***

**Navigation** : [Predecessor: [07-Committees-Core]] | [Index](../README.md)